In [1]:
import numpy as np
from tensorflow.keras.models import load_model

# -----------------------------
# Config
# -----------------------------
MODEL_PATH  = "../../../models/Model4/model4_cnn.keras"
X_TEST_PATH = "../../../data/final test/Behaviour_final test_X.npy"
Y_TEST_PATH = "../../../data/final test/Behaviour_final test_y.npy"

# -----------------------------
# Helpers
# -----------------------------
def to_int(label):
    if isinstance(label, (int, np.integer)):
        return int(label)
    s = str(label).strip().lower()
    if "steady"   in s: return 0
    if "acceler"  in s: return 1
    if "deceler"  in s or "decelar" in s: return 2
    raise ValueError(f"Unrecognized label: {label}")

def add_delta_rpm(X):
    # If already (N,2,5), return as-is
    if X.ndim == 3 and X.shape[1:] == (2, 5):
        return X.astype("float32")
    # Expect (N,2,4): add delta RPM as 5ª feature
    if X.ndim == 3 and X.shape[1:] == (2, 4):
        X = X.astype("float32")
        rpm = X[:, :, 2]                                   # (N,2)
        delta = (rpm[:, 1] - rpm[:, 0]).reshape(-1, 1)     # (N,1)
        delta_tiled = np.repeat(delta[:, np.newaxis, :], 2, axis=1)  # (N,2,1)
        return np.concatenate([X, delta_tiled], axis=2)    # (N,2,5)
    raise ValueError(f"Expected X shape (N,2,4) or (N,2,5), got {X.shape}")

# -----------------------------
# Load data
# -----------------------------
X_test_raw = np.load(X_TEST_PATH, allow_pickle=True)
y_test_raw = np.load(Y_TEST_PATH, allow_pickle=True)

X_test = add_delta_rpm(X_test_raw)
y_test = np.array([to_int(v) for v in y_test_raw], dtype=np.int64)

# -----------------------------
# Evaluate
# -----------------------------
model = load_model(MODEL_PATH)
loss, acc = model.evaluate(X_test, y_test, verbose=1)
print(f"✅ Final Test Accuracy: {acc:.4f} | Loss: {loss:.4f}")


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9955 - loss: 0.0122  
✅ Final Test Accuracy: 0.9955 | Loss: 0.0122
